# Employee Attrition Analytics & Prediction

This notebook covers the exploratory data analysis (EDA) and pipeline modeling steps to predict employee attrition. We explore structural drivers of turnover and train a machine learning model to flag high-risk departures.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting styles
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Load data
df = pd.read_csv('../dataset/employee_attrition.csv')
print(f"Data loaded successfully. Dimensions: {df.shape}")
df.head()

### Initial Data Checks

In [ ]:
print("Missing Values:", df.isnull().sum().sum())
print("Duplicate Rows:", df.duplicated().sum())
print("\nDistribution of key numerical statistics:")
df[['Age', 'MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears']].describe()

### Exploratory Data Analysis

In [ ]:
# 1. Target Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Attrition', palette='Set2')
plt.title('Overall Employee Attrition Distribution')
plt.show()

In [ ]:
# 2. Monthly Income vs Attrition Boxplot
sns.boxplot(data=df, x='Attrition', y='MonthlyIncome', palette='Set2')
plt.title('Monthly Income Distribution by Attrition State')
plt.ylabel('Monthly Income ($)')
plt.show()

In [ ]:
# 3. Overtime Status vs Attrition count
sns.countplot(data=df, x='OverTime', hue='Attrition', palette='Set2')
plt.title('Attrition Rate based on Overtime Status')
plt.show()

In [ ]:
# 4. Job Satisfaction level vs Attrition
sns.countplot(data=df, x='JobSatisfaction', hue='Attrition', palette='Set2')
plt.title('Attrition across Job Satisfaction Levels (1=Low, 4=High)')
plt.xlabel('Job Satisfaction Level')
plt.show()

### Pipeline Modeling
We encapsulate column encoding, feature scaling, and estimator fitting into a Scikit-Learn `Pipeline`. This prevents data leakage and ensures clean deployments.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Drop uninformative metadata features
drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
X = df.drop(columns=[c for c in drop_cols if c in df.columns] + ['Attrition'])
y = df['Attrition'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)

num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Model with class imbalance handled
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)
y_probs = model_pipeline.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_probs):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

### Feature Importances

In [ ]:
cat_encoder = model_pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_names = cat_encoder.get_feature_names_out(cat_cols).tolist()
all_features = num_cols + cat_names
importances = model_pipeline.named_steps['classifier'].feature_importances_

importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Visualizing Top 15 importances
plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df.head(15), x='Importance', y='Feature', palette='viridis')
plt.title('Top 15 Predictors of Employee Attrition')
plt.xlabel('Importance Weight')
plt.tight_layout()
plt.show()